# YOLO Inference on UNIFIED_MEDIA

## Goal
This notebook runs the trained YOLOv8 model on the unified media dataset to detect wild boars. 
The output will be a CSV file mapping each image to the number of boars detected, which is essential for the downstream Royle-Nichols (RN) population estimation model.

## Inputs
1. **`yolo26n.pt`** — Trained YOLOv8 model.
2. **`unified_media/unified_manifest.csv`** — List of all images to process.

## Outputs
1. **`eda_outputs/unified_detections.csv`** — Detection counts and confidence levels.

In [9]:
import os
from pathlib import Path
import pandas as pd
from tqdm.auto import tqdm
from ultralytics import YOLO
import torch

# Setup paths
REPO = Path("/home/chengwei/boartraining/HackMITChina2026")
MODEL_PATH = REPO / "yolo26n.pt"
MANIFEST_PATH = REPO / "unified_media" / "unified_manifest.csv"
IMAGES_DIR = REPO / "unified_media" / "images"
OUT_DIR = REPO / "eda_outputs"
OUT_CSV = OUT_DIR / "unified_detections.csv"

OUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Using model: {MODEL_PATH}")
print(f"Processing manifest: {MANIFEST_PATH}")

Using model: /home/chengwei/boartraining/HackMITChina2026/yolo26n.pt
Processing manifest: /home/chengwei/boartraining/HackMITChina2026/unified_media/unified_manifest.csv


In [10]:
# Load Model
model = YOLO(MODEL_PATH)

# Check device
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Running on device: {device}")

Running on device: cuda


In [11]:
# Load Manifest
df_manifest = pd.read_csv(MANIFEST_PATH)
print(f"Total images to process: {len(df_manifest)}")

# Construct full paths for inference
df_manifest.head()

Total images to process: 25701


,unified_relpath,unified_path,source_path,source_media_type,station_id,station_raw,original_filename,source_suffix,video_middle_frame_index,source_size_bytes
0,images/B1__img__c14a239d4be0__IMG_0062.jpg,/home/chengwei/boartraining/HackMITChina2026/u...,/home/chengwei/boartraining/HackMITChina2026/a...,image,B1,B1,IMG_0062.jpg,.jpg,NaN,74393
1,images/B1__img__efaf9a2cb2bf__IMG_0057.jpg,/home/chengwei/boartraining/HackMITChina2026/u...,/home/chengwei/boartraining/HackMITChina2026/a...,image,B1,B1,IMG_0057.jpg,.jpg,NaN,17935
2,images/B1__img__9abcb6d01f91__IMG_0031.jpg,/home/chengwei/boartraining/HackMITChina2026/u...,/home/chengwei/boartraining/HackMITChina2026/a...,image,B1,B1,IMG_0031.jpg,.jpg,NaN,118127
3,images/B1__img__0c76a03af4f4__IMG_0039.jpg,/home/chengwei/boartraining/HackMITChina2026/u...,/home/chengwei/boartraining/HackMITChina2026/a...,image,B1,B1,IMG_0039.jpg,.jpg,NaN,116939
4,images/B1__img__618a041f30fd__IMG_0021.jpg,/home/chengwei/boartraining/HackMITChina2026/u...,/home/chengwei/boartraining/HackMITChina2026/a...,image,B1,B1,IMG_0021.jpg,.jpg,NaN,98010


In [12]:
# Run Inference in batches
results_list = []

# Batch size for inference
BATCH_SIZE = 32

for i in tqdm(range(0, len(df_manifest), BATCH_SIZE)):
    batch_df = df_manifest.iloc[i : i + BATCH_SIZE]

    # Build absolute paths for prediction
    img_paths = [str(REPO / p) if not Path(p).is_absolute() else p for p in batch_df["unified_path"]]

    # Run YOLO
    batch_results = model.predict(img_paths, conf=0.25, verbose=False, device=device)

    for idx, res in enumerate(batch_results):
        row = batch_df.iloc[idx]

        boxes = res.boxes
        boar_count = len(boxes)
        max_conf = float(boxes.conf.max()) if boar_count > 0 else 0.0

        results_list.append(
            {
                "unified_path": row["unified_path"],
                "unified_relpath": row.get("unified_relpath", ""),
                "source_path": row.get("source_path", ""),
                "station_id": row.get("station_id", "unknown"),
                "boar_count": boar_count,
                "max_conf": max_conf,
            }
        )

# Create results DataFrame
key_cols = [
    "unified_path",
    "unified_relpath",
    "source_path",
    "station_id",
    "boar_count",
    "max_conf",
]
df_results = pd.DataFrame(results_list)
df_results = df_results[key_cols]
df_results.head()

  0%|          | 0/804 [00:00<?, ?it/s]

OSError: image file is truncated (22 bytes not processed)

In [ ]:
# Save results
df_results.to_csv(OUT_CSV, index=False)
print(f"Detections saved to {OUT_CSV}")

# Quick Summary
print(f"Total detections: {df_results['boar_count'].sum()}")
print(f"Images with at least one boar: {(df_results['boar_count'] > 0).sum()}")
print("Columns:", list(df_results.columns))